In [1]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt

# Load master dataset
df = pd.read_csv(r"D:\Profession\My_Space\DS\ML\My_Projects\Vegetation_Encroachment_Risk_Detection_California_Transmission_Lines\Data\master_dataset.csv")

print(f"Dataset shape: {df.shape}")
print(df[['kV', 'NDVI', 'slope', 'landcover', 'risk_label']].describe())

Dataset shape: (6675, 25)
                kV         NDVI        slope    landcover   risk_label
count  6668.000000  6675.000000  6675.000000  6674.000000  6675.000000
mean    115.669691     0.286368     5.413038    41.577914     0.004494
std      83.077544     0.223132     6.219739    23.740189     0.066894
min      33.000000    -1.000000     0.000000    11.000000     0.000000
25%      66.000000     0.118473     1.531264    23.000000     0.000000
50%      69.000000     0.243767     2.999637    24.000000     0.000000
75%     115.000000     0.412944     5.990175    71.000000     0.000000
max     500.000000     0.915518    44.765700    95.000000     1.000000


In [2]:
# Prepare features
features = ['kV', 'NDVI', 'slope', 'landcover']
target = 'risk_label'

# Drop rows with missing values
df_clean = df[features + [target]].dropna()
print(f"Clean dataset rows: {len(df_clean)}")

X = df_clean[features]
y = df_clean[target]

# Split train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training rows: {len(X_train)}")
print(f"Testing rows: {len(X_test)}")
print(f"High risk in train: {y_train.sum()}")
print(f"High risk in test: {y_test.sum()}")

Clean dataset rows: 6667
Training rows: 5333
Testing rows: 1334
High risk in train: 24
High risk in test: 6


In [3]:
# Handle class imbalance
scale = int((y_train == 0).sum() / (y_train == 1).sum())
print(f"Scale pos weight: {scale}")

# Train XGBoost
model = XGBClassifier(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    scale_pos_weight=scale,
    random_state=42,
    eval_metric='logloss'
)

model.fit(X_train, y_train)

# Evaluate
y_pred = model.predict(X_test)

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Scale pos weight: 221

Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.98      0.99      1328
           1       0.09      0.33      0.14         6

    accuracy                           0.98      1334
   macro avg       0.54      0.66      0.57      1334
weighted avg       0.99      0.98      0.99      1334

Confusion Matrix:
[[1308   20]
 [   4    2]]


In [6]:
# Lower threshold to catch more high risk segments
y_pred_proba = model.predict_proba(X_test)[:, 1]

# Try threshold 0.3 instead of default 0.5
y_pred_adjusted = (y_pred_proba >= 0.3).astype(int)

print("Adjusted Classification Report:")
print(classification_report(y_test, y_pred_adjusted))

print("Adjusted Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_adjusted))

Adjusted Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.97      0.98      1328
           1       0.07      0.50      0.13         6

    accuracy                           0.97      1334
   macro avg       0.54      0.74      0.56      1334
weighted avg       0.99      0.97      0.98      1334

Adjusted Confusion Matrix:
[[1290   38]
 [   3    3]]


In [8]:
import joblib

# Save model with threshold 0.3
joblib.dump(model, 
    r"D:\Profession\My_Space\DS\ML\My_Projects\Vegetation_Encroachment_Risk_Detection_California_Transmission_Lines\outputs\cadvegwatch_model.pkl"
)

# Add predictions to full dataset
df_clean['risk_score'] = model.predict_proba(X)[:, 1]
df_clean['risk_category'] = pd.cut(
    df_clean['risk_score'],
    bins=[0, 0.3, 0.6, 1.0],
    labels=['Low', 'Medium', 'High']
)

print("Model saved!")
print(df_clean['risk_category'].value_counts())

Model saved!
risk_category
Low       6455
Medium     112
High       100
Name: count, dtype: int64


In [9]:
# Save predictions
df_clean.to_csv(
    r"D:\Profession\My_Space\DS\ML\My_Projects\Vegetation_Encroachment_Risk_Detection_California_Transmission_Lines\outputs\predictions.csv",
    index=False
)

print("Predictions saved!")
print(df_clean[['kV', 'NDVI', 'slope', 'landcover', 'risk_score', 'risk_category']].head(10))

Predictions saved!
      kV      NDVI     slope  landcover  risk_score risk_category
0  115.0  0.261563  2.521554       23.0    0.002637           Low
1  115.0  0.502851  2.988725       22.0    0.007254           Low
2  115.0  0.260692  6.870550       23.0    0.035000           Low
3  115.0  0.050684  0.927410       24.0    0.000318           Low
4   34.0  0.602863  4.339175       52.0    0.000524           Low
5   69.0  0.330989  2.416575       24.0    0.002721           Low
6   69.0  0.183662  2.995485       23.0    0.009521           Low
7   69.0  0.018874  1.451196       24.0    0.000944           Low
8   69.0  0.158762  4.985468       24.0    0.000957           Low
9   69.0  0.068601  3.868407       24.0    0.000943           Low


In [14]:
print(df_clean['risk_category'].value_counts())
print(f"\nHigh risk segments: {(df_clean['risk_category'] == 'High').sum()}")

risk_category
Low       6455
Medium     112
High       100
Name: count, dtype: int64

High risk segments: 100
